In [3]:
import sys

!{sys.executable} -m pip install datasets pillow pandas tqdm numpy matplotlib seaborn opencv-python scikit-learn

  Using cached datasets-4.8.5-py3-none-any.whl.metadata (19 kB)
  Using cached huggingface_hub-1.17.0-py3-none-any.whl.metadata (14 kB)
  Using cached click-8.4.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached hf_xet-1.5.0-cp37-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
Using cached datasets-4.8.5-py3-none-any.whl (528 kB)
Using cached huggingface_hub-1.17.0-py3-none-any.whl (671 kB)
Using cached hf_xet-1.5.0-cp37-abi3-win_amd64.whl (4.0 MB)
Using cached dill-0.4.1-py3-none-any.whl (120 kB)
Using cached click-8.4.1-py3-none-any.whl (116 kB)

   ----- ---------------------------------- 1/7 [hf-xet]
  Attempting uninstall: dill
   ----- ---------------------------------- 1/7 [hf-xet]
    Found existing installation: dill 0.4.0
   ----- ---------------------------------- 1/7 [hf-xet]
    Uninstalling dill-0.4.0:
   ----- ---------------------------------- 1/7 [hf-xet]
      Successfully uninstalled dill-0.4.0
   ----- --------------

In [1]:
import os
import json
import random
import shutil
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from datasets import load_dataset

print("Libraries loaded successfully")


Libraries loaded successfully


In [2]:
SELECTED_CLASSES = {
    'person': 0,
    'bicycle': 1,
    'car': 2,
    'motorcycle': 3,
    'airplane': 4,
    'bus': 5,
    'truck': 7,
    'traffic light': 9,
    'stop sign': 11,
    'bench': 13,
    'bird': 14,
    'cat': 15,
    'dog': 16,
    'horse': 17,
    'cow': 19,
    'elephant': 20,
    'bottle': 39,
    'cup': 41,
    'bowl': 45,
    'pizza': 53,
    'cake': 55,
    'chair': 56,
    'couch': 57,
    'potted plant': 58,
    'bed': 59
}

IMAGES_PER_CLASS = 100
BASE_DIR = "../smartvision_dataset"

print("Number of classes:", len(SELECTED_CLASSES))

Number of classes: 25


In [3]:
dataset = load_dataset(
    "detection-datasets/coco",
    split="train",
    streaming=True
)

print("COCO dataset loaded successfully")

README.md:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

C:\Users\Home\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Home\.cache\huggingface\hub\datasets--detection-datasets--coco. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Resolving data files:   0%|          | 0/40 [00:00<?, ?it/s]

dataset_infos.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

COCO dataset loaded successfully


In [4]:
class_images = {class_name: [] for class_name in SELECTED_CLASSES.keys()}
class_counts = {class_name: 0 for class_name in SELECTED_CLASSES.keys()}

total_collected = 0
images_processed = 0
max_iterations = 50000

print("Starting image collection...")

for idx, item in enumerate(dataset):
    images_processed += 1

    if images_processed % 1000 == 0:
        print(f"Processed {images_processed} images | Collected {total_collected}/2500")

    if images_processed >= max_iterations:
        print("Reached safety limit")
        break

    if all(count >= IMAGES_PER_CLASS for count in class_counts.values()):
        print("Collected 100 images for all classes")
        break

    annotations = item["objects"]
    categories = annotations["category"]

    for cat_id in categories:
        for class_name, class_id in SELECTED_CLASSES.items():
            if cat_id == class_id and class_counts[class_name] < IMAGES_PER_CLASS:
                class_images[class_name].append({
                    "image": item["image"],
                    "annotations": item["objects"],
                    "idx": images_processed
                })

                class_counts[class_name] += 1
                total_collected += 1

                if total_collected % 100 == 0:
                    print(f"Collected {total_collected}/2500 images")

                break

print("Collection complete")
print("Images processed:", images_processed)
print("Images collected:", total_collected)

for class_name, count in sorted(class_counts.items()):
    print(class_name, ":", count)

Starting image collection...
Collected 100/2500 images
Collected 200/2500 images
Collected 300/2500 images
Collected 400/2500 images
Collected 500/2500 images
Collected 600/2500 images
Collected 700/2500 images
Collected 800/2500 images
Collected 900/2500 images
Collected 1000/2500 images
Collected 1100/2500 images
Collected 1200/2500 images
Collected 1300/2500 images
Collected 1400/2500 images
Collected 1500/2500 images
Collected 1600/2500 images
Processed 1000 images | Collected 1642/2500
Collected 1700/2500 images
Collected 1800/2500 images
Collected 1900/2500 images
Collected 2000/2500 images
Collected 2100/2500 images
Collected 2200/2500 images
Processed 2000 images | Collected 2273/2500
Collected 2300/2500 images
Collected 2400/2500 images
Processed 3000 images | Collected 2443/2500
Processed 4000 images | Collected 2459/2500
Processed 5000 images | Collected 2474/2500
Collected 2500/2500 images
Collected 100 images for all classes
Collection complete
Images processed: 5979
Image

In [5]:
# Create folder structure

os.makedirs(BASE_DIR, exist_ok=True)

# Classification folders
for split in ["train", "val", "test"]:
    for class_name in SELECTED_CLASSES.keys():
        os.makedirs(
            f"{BASE_DIR}/classification/{split}/{class_name}",
            exist_ok=True
        )

# Detection folders
os.makedirs(f"{BASE_DIR}/detection/images", exist_ok=True)
os.makedirs(f"{BASE_DIR}/detection/labels", exist_ok=True)

print("Folder structure created successfully")

Folder structure created successfully


In [6]:
metadata = {
    'total_images': 0,
    'classes': {},
    'splits': {'train': 0, 'val': 0, 'test': 0}
}

train_data = {}
val_data = {}
test_data = {}

for class_name in SELECTED_CLASSES.keys():

    all_items = class_images[class_name]

    n = len(all_items)

    train_split = int(0.7 * n)
    val_split = int(0.85 * n)

    train_data[class_name] = all_items[:train_split]
    val_data[class_name] = all_items[train_split:val_split]
    test_data[class_name] = all_items[val_split:]

    metadata['classes'][class_name] = {
        'train': len(train_data[class_name]),
        'val': len(val_data[class_name]),
        'test': len(test_data[class_name]),
        'total': len(all_items)
    }

    metadata['splits']['train'] += len(train_data[class_name])
    metadata['splits']['val'] += len(val_data[class_name])
    metadata['splits']['test'] += len(test_data[class_name])

print("Dataset split completed")

Dataset split completed


In [7]:
classification_stats = {'train': 0, 'val': 0, 'test': 0}

for split_name, split_data in [('train', train_data), ('val', val_data), ('test', test_data)]:
    print(f"Saving {split_name} classification images...")

    for class_name, items in tqdm(split_data.items()):
        class_folder = f"{BASE_DIR}/classification/{split_name}/{class_name}"
        class_id = SELECTED_CLASSES[class_name]

        for img_idx, item in enumerate(items):
            img = item['image']
            annotations = item['annotations']
            bboxes = annotations['bbox']
            categories = annotations['category']

            for bbox, cat_id in zip(bboxes, categories):
                if cat_id == class_id:
                    x, y, w, h = bbox

                    cropped_img = img.crop((x, y, x + w, y + h))
                    cropped_img = cropped_img.resize((224, 224), Image.LANCZOS)

                    img_filename = f"{class_name}_{split_name}_{img_idx:04d}.jpg"
                    img_path = os.path.join(class_folder, img_filename)
                    cropped_img.save(img_path, quality=95)

                    classification_stats[split_name] += 1
                    break

print("Classification images saved")
print(classification_stats)


Saving train classification images...


100%|██████████████████████████████████████████████████████████████████████████████████| 25/25 [00:17<00:00,  1.41it/s]


Saving val classification images...


100%|██████████████████████████████████████████████████████████████████████████████████| 25/25 [00:03<00:00,  7.15it/s]


Saving test classification images...


100%|██████████████████████████████████████████████████████████████████████████████████| 25/25 [00:03<00:00,  7.88it/s]

Classification images saved
{'train': 1750, 'val': 375, 'test': 375}


In [9]:
print("Creating YOLO detection dataset...")

Creating YOLO detection dataset...


In [10]:
# PART B: SAVE DETECTION IMAGES (YOLO FORMAT)

print("="*70)
print("📁 PART B: Saving Detection Images & Annotations...")
print("   Format: Full images with YOLO .txt labels\n")

detection_stats = {'images': 0, 'annotations': 0, 'objects': 0}

# COCO to YOLO class mapping
coco_to_yolo = {class_id: idx for idx, class_id in enumerate(SELECTED_CLASSES.values())}

# Combine train + val for detection
all_detection_data = []
for class_name in SELECTED_CLASSES.keys():
    all_detection_data.extend(train_data.get(class_name, []))
    all_detection_data.extend(val_data.get(class_name, []))

print(f"📊 Total detection images: {len(all_detection_data)}\n")

# Save images and create YOLO labels
for img_idx, item in enumerate(tqdm(all_detection_data, desc="Saving detection data")):

    img = item['image']
    img_width, img_height = img.size

    # Save full image
    img_filename = f"image_{img_idx:06d}.jpg"
    img_path = os.path.join(f"{BASE_DIR}/detection/images", img_filename)
    img.save(img_path, quality=95)
    detection_stats['images'] += 1

    # Get annotations
    annotations = item['annotations']
    bboxes = annotations['bbox']
    categories = annotations['category']

    # Create YOLO annotation
    label_filename = f"image_{img_idx:06d}.txt"
    label_path = os.path.join(f"{BASE_DIR}/detection/labels", label_filename)

    yolo_annotations = []
    objects_count = 0

    for bbox, cat_id in zip(bboxes, categories):
        if cat_id in coco_to_yolo:
            x, y, w, h = bbox

            # Convert to YOLO format (normalized)
            x_center = (x + w/2) / img_width
            y_center = (y + h/2) / img_height
            w_norm = w / img_width
            h_norm = h / img_height

            yolo_class_id = coco_to_yolo[cat_id]
            yolo_line = f"{yolo_class_id} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}"
            yolo_annotations.append(yolo_line)
            objects_count += 1

    # Save label file
    if yolo_annotations:
        with open(label_path, 'w') as f:
            f.write('\n'.join(yolo_annotations))
        detection_stats['annotations'] += 1
        detection_stats['objects'] += objects_count

print()
print("="*70)
print("✅ DETECTION DATASET CREATED!")
print("="*70)
print(f"📊 Images:     {detection_stats['images']}")
print(f"📊 Labels:     {detection_stats['annotations']}")
print(f"📊 Objects:    {detection_stats['objects']}")
print(f"📊 Avg/image:  {detection_stats['objects']/detection_stats['images']:.2f}")
print()

📁 PART B: Saving Detection Images & Annotations...
   Format: Full images with YOLO .txt labels

📊 Total detection images: 2125



Saving detection data: 100%|███████████████████████████████████████████████████████| 2125/2125 [00:40<00:00, 52.90it/s]


✅ DETECTION DATASET CREATED!
📊 Images:     2125
📊 Labels:     2125
📊 Objects:    21578
📊 Avg/image:  10.15



In [11]:
# PART C: CREATE YOLO CONFIG FILE

print("📝 Creating YOLO configuration file...\n")

yaml_content = f"""# SmartVision Dataset - YOLOv8 Configuration
path: {os.path.abspath(BASE_DIR)}/detection
train: images
val: images

names:
  0: person
  1: bicycle
  2: car
  3: motorcycle
  4: airplane
  5: bus
  6: train
  7: truck
  8: traffic light
  9: stop sign
  10: bench
  11: bird
  12: cat
  13: dog
  14: horse
  15: cow
  16: elephant
  17: bottle
  18: cup
  19: bowl
  20: pizza
  21: cake
  22: chair
  23: couch
  24: potted plant
  25: bed

nc: 26
"""

yaml_path = f"{BASE_DIR}/detection/data.yaml"
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print(f"✅ Created: {yaml_path}\n")

📝 Creating YOLO configuration file...

✅ Created: ../smartvision_dataset/detection/data.yaml



In [12]:
# PART D: SAVE METADATA

print("📊 Saving metadata...\n")

metadata['classification'] = classification_stats
metadata['detection'] = detection_stats
metadata['dataset_path'] = os.path.abspath(BASE_DIR)

metadata_path = f"{BASE_DIR}/dataset_metadata.json"
with open(metadata_path, 'w') as f:
    json.dump(metadata, indent=2, fp=f)

print(f"✅ Saved: {metadata_path}\n")

📊 Saving metadata...

✅ Saved: ../smartvision_dataset/dataset_metadata.json



In [13]:
print("="*70)
print("🎉 DATASET SETUP COMPLETE!")
print("="*70)
print()
print(f"📁 Location: {os.path.abspath(BASE_DIR)}")
print()
print("📂 Classification Dataset:")
print(f"   ├─ Train:  {classification_stats['train']} images (70%)")
print(f"   ├─ Val:    {classification_stats['val']} images (15%)")
print(f"   ├─ Test:   {classification_stats['test']} images (15%)")
print(f"   └─ Total:  {sum(classification_stats.values())} cropped images (224x224)")
print()
print("📂 Detection Dataset:")
print(f"   ├─ Images: {detection_stats['images']} full images")
print(f"   ├─ Labels: {detection_stats['annotations']} YOLO .txt files")
print(f"   └─ Objects: {detection_stats['objects']} annotated objects")
print()
print("="*70)
print("✅ LEARNERS CAN NOW START:")
print("="*70)
print("Step 7:  Exploratory Data Analysis (EDA)")
print("Step 8:  Train Classification Models")
print("Step 9:  Train YOLO Detection Model")
print("Step 10: Build Streamlit Application")
print("Step 11: Deploy to Hugging Face Spaces")
print("="*70)

🎉 DATASET SETUP COMPLETE!

📁 Location: C:\Users\Home\SmartVision_AI\smartvision_dataset

📂 Classification Dataset:
   ├─ Train:  1750 images (70%)
   ├─ Val:    375 images (15%)
   ├─ Test:   375 images (15%)
   └─ Total:  2500 cropped images (224x224)

📂 Detection Dataset:
   ├─ Images: 2125 full images
   ├─ Labels: 2125 YOLO .txt files
   └─ Objects: 21578 annotated objects

✅ LEARNERS CAN NOW START:
Step 7:  Exploratory Data Analysis (EDA)
Step 8:  Train Classification Models
Step 9:  Train YOLO Detection Model
Step 10: Build Streamlit Application
Step 11: Deploy to Hugging Face Spaces


In [14]:
import os
import random

label_dir = "../smartvision_dataset/detection/labels"

sample_file = random.choice(os.listdir(label_dir))

print("Checking:", sample_file)

with open(os.path.join(label_dir, sample_file)) as f:
    print(f.read())

Checking: image_001488.txt
17 0.862484 0.422240 0.617531 0.340187
17 0.796242 0.390948 0.573297 0.311938
17 1.136703 0.404031 0.803562 0.329354
17 1.233984 0.382562 0.868250 0.305542
17 0.683039 0.311563 0.484453 0.254000
17 0.470156 0.255333 0.338312 0.188500
17 1.051930 0.367823 0.744203 0.291687
17 0.964719 0.372990 0.686250 0.290146
17 0.756062 0.290250 0.537594 0.252083
17 0.464203 0.315135 0.341844 0.251729
17 0.810477 0.265281 0.554641 0.211229
17 0.629430 0.347687 0.449859 0.285667


In [15]:
import os

label_dir = "../smartvision_dataset/detection/labels"

count = 0

for file in os.listdir(label_dir):

    if not file.endswith(".txt"):
        continue

    with open(os.path.join(label_dir, file)) as f:

        for line in f:

            vals = line.strip().split()

            if len(vals) != 5:
                continue

            _, x, y, w, h = map(float, vals)

            if max(x, y, w, h) > 1:

                count += 1
                print(file, line.strip())

                if count >= 20:
                    break

    if count >= 20:
        break

print("\nInvalid labels found:", count)

image_000001.txt 0 0.485512 1.036200 0.344567 0.735560
image_000001.txt 23 0.887336 1.325800 0.715512 0.946840
image_000001.txt 0 1.133727 1.027630 0.774856 0.721660
image_000001.txt 0 0.761365 1.026520 0.520105 0.715560
image_000001.txt 0 1.384619 1.009420 0.948976 0.685240
image_000002.txt 0 0.485512 1.036200 0.344567 0.735560
image_000002.txt 23 0.887336 1.325800 0.715512 0.946840
image_000002.txt 0 1.133727 1.027630 0.774856 0.721660
image_000002.txt 0 0.761365 1.026520 0.520105 0.715560
image_000002.txt 0 1.384619 1.009420 0.948976 0.685240
image_000003.txt 0 0.485512 1.036200 0.344567 0.735560
image_000003.txt 23 0.887336 1.325800 0.715512 0.946840
image_000003.txt 0 1.133727 1.027630 0.774856 0.721660
image_000003.txt 0 0.761365 1.026520 0.520105 0.715560
image_000003.txt 0 1.384619 1.009420 0.948976 0.685240
image_000004.txt 0 0.485512 1.036200 0.344567 0.735560
image_000004.txt 23 0.887336 1.325800 0.715512 0.946840
image_000004.txt 0 1.133727 1.027630 0.774856 0.721660
image_

In [16]:
import os

label_dir = "../smartvision_dataset/detection/labels"

fixed_count = 0
removed_lines = 0

for file in os.listdir(label_dir):
    if not file.endswith(".txt"):
        continue

    file_path = os.path.join(label_dir, file)
    valid_lines = []

    with open(file_path, "r") as f:
        lines = f.readlines()

    for line in lines:
        values = line.strip().split()

        if len(values) != 5:
            removed_lines += 1
            continue

        cls, x, y, w, h = values
        x, y, w, h = float(x), float(y), float(w), float(h)

        # keep only valid YOLO boxes
        if 0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1:
            valid_lines.append(
                f"{cls} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n"
            )
        else:
            removed_lines += 1

    with open(file_path, "w") as f:
        f.writelines(valid_lines)

    fixed_count += 1

print("Label files checked:", fixed_count)
print("Invalid label lines removed:", removed_lines)

Label files checked: 2125
Invalid label lines removed: 8741


In [17]:
label_dir = "../smartvision_dataset/detection/labels"

count = 0

for file in os.listdir(label_dir):
    if not file.endswith(".txt"):
        continue

    with open(os.path.join(label_dir, file)) as f:
        for line in f:
            vals = line.strip().split()

            if len(vals) != 5:
                continue

            _, x, y, w, h = map(float, vals)

            if max(x, y, w, h) > 1:
                count += 1

print("Invalid labels found:", count)

Invalid labels found: 0
